In [5]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

@tool
def search_duckduckgo(query: str) -> str:

    """This tool searches the latest news on DuckDuckGo for the given query and returns the results."""
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)

@tool
def wiki_tool(query: str):

    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

@tool
def personal_info(name: str):

    """Use this tool to get personal information about Alice, Bob, or Charlie.
    """

    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person.")

tools = [search_duckduckgo, wiki_tool, personal_info]

llm_with_tools = llm_openai.bind_tools(tools)
class graph_schema(TypedDict):
    messages: List

def llm_node(state: graph_schema) -> graph_schema:

    messages = state['messages']

    # Prompt template for the LLM, including system instructions and the human input
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant that can use tools to answer questions."),
            ("human", "{input}")
        ]
    )

    # LLM With Tools
    # llm_with_tools

    chain = prompt | llm_with_tools

    response = chain.invoke({"input": messages})

    # Update the state with the new message
    state['messages'] = messages + [response]

    return state

def tool_node(state: graph_schema) -> graph_schema:

    messages = state['messages']

    tools_by_name = {tool.name: tool for tool in tools}

    tool_results = []

    for tool_call in messages[-1].tool_calls:

        tool = tools_by_name[tool_call["name"]]

        observation = tool.invoke(tool_call["args"])

        tool_results.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))

    # Update the state with the tool results
    state['messages'] = messages + tool_results

    return state

def if_tool_call(state: graph_schema) -> str:

    last_message = state['messages'][-1]

    if last_message.tool_calls:
        return "tool_node"
    else:
        return "end"

graph=StateGraph(graph_schema)

graph.add_node("llm_node",llm_node)
graph.add_node("tool_node",tool_node)

graph.add_edge(START,"llm_node")
graph.add_conditional_edges("llm_node",if_tool_call,{"tool_node":"tool_node","end":END})
graph.add_edge("tool_node","llm_node")
graph.add_edge("llm_node",END)

f_graph=graph.compile()

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())

result=f_graph.invoke({"messages": [HumanMessage(content="langgraph what is??")]})

print(result)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm_node(llm_node)
	tool_node(tool_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm_node;
	llm_node -. &nbsp;end&nbsp; .-> __end__;
	llm_node -.-> tool_node;
	tool_node --> llm_node;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

{'messages': [HumanMessage(content='langgraph what is??', additional_kwargs={}, response_metadata={}), AIMessage(content='**LangGraph** is an open‑source Python library (maintained by the LangChain team) that makes it easy to build **state‑driven, graph‑structured workflows** for large‑language‑model (LLM) applications.\n\n---\n\n## Why it exists\n\nWhen you build an LLM‑powered app you often need more than a single “prompt → response” call. Real‑world agents require:\n\n| Need | Typical solution without LangGraph | What LangGraph adds |\n|------|-----------------------------------|--------------